In [3]:
!pip install gensim
!pip install tensorflow

In [4]:
from google.colab import files
uploaded=files.upload()

Saving test.csv to test.csv


In [5]:
import pandas as pd
import numpy as np
import tensorflow as tf

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense,Flatten, Input, Dropout
from sklearn.model_selection import train_test_split
from gensim.parsing.preprocessing import preprocess_string
from nltk import sent_tokenize

In [7]:
df = pd.read_csv('test.csv').iloc[7000:18000]

In [8]:
df.drop_duplicates(inplace=True)


In [9]:
df['text'] = df['text'].apply(preprocess_string)

In [11]:
doc=[]
for i in df['text']:
  doc.append(i)

In [12]:
import gensim
model=gensim.models.Word2Vec(window=5,min_count=2,vector_size=300)

In [13]:
model.build_vocab(doc)
model.train(doc,total_examples=model.corpus_count,epochs=model.epochs)

(5217841, 5588905)

In [14]:
def doc_vec(x):
    j=[i for i in x if i in model.wv.index_to_key]
    return np.mean(model.wv[j],axis=0)

In [15]:
X=[]
for k in df['text']:
    X.append(doc_vec(k))
X=np.array(X)


In [16]:
y=df['label']

In [17]:
nn = Sequential([
    Dense(128, activation='relu', input_shape=(300,)),
    Dropout(0.3),

    Dense(64, activation='relu'),
    Dropout(0.3),

    Dense(1, activation='sigmoid')
])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [18]:
X_train, X_test, y_train, y_test=train_test_split(X,y,test_size=0.2,random_state=7)

In [19]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [20]:
from tensorflow.keras.optimizers import Adam
nn.compile(optimizer='adam',loss='binary_crossentropy',metrics=['accuracy'])
nn.fit(X_train,y_train,epochs=15)

Epoch 1/15
275/275 ━━━━━━━━━━━━━━━━━━━━ 5s 8ms/step - accuracy: 0.7062 - loss: 0.5816
Epoch 2/15
275/275 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.7445 - loss: 0.5186
Epoch 3/15
275/275 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.7481 - loss: 0.5142
Epoch 4/15
275/275 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.7552 - loss: 0.4995
Epoch 5/15
275/275 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.7595 - loss: 0.4942
Epoch 6/15
275/275 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.7697 - loss: 0.4868
Epoch 7/15
275/275 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.7672 - loss: 0.4868
Epoch 8/15
275/275 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.7624 - loss: 0.4830
Epoch 9/15
275/275 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.7676 - loss: 0.4778
Epoch 10/15
275/275 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.7754 - loss: 0.4752
Epoch 11/15
275/275 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.7764 - loss: 0.4733
Epoch 12/15
275/275 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step

In [21]:
nn.evaluate(X_test,y_test)

69/69 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - accuracy: 0.7762 - loss: 0.4772


[0.4771656095981598, 0.7762078642845154]

In [22]:
from sklearn.metrics import f1_score

y_pred = (nn.predict(X_test) > 0.5).astype(int)
print(f1_score(y_test, y_pred))

69/69 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step
0.7645083932853717
